In [ ]:
# ============================================================
# CELL 0 — Install dependencies (run once)
# ============================================================
!pip install torch-geometric -q


In [ ]:
# ============================================================
# CELL 1 — Imports & Config
# ============================================================
import json, pickle, random, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, SAGPooling

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)

# ---- Hyper-params ----
HIDDEN_DIM   = 64      # GAT hidden dim
HEADS        = 2       # GATConv attention heads
EMBED_DIM    = 64      # GCN / predictor dim
GAT_BATCH    = 32      # proteins per GAT chunk (lower = less VRAM)
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 200     # <-- 200 epochs
SEED         = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")


In [ ]:
# ============================================================
# CELL 2 — Load data
# ============================================================
BASE = "/kaggle/input/datasets/shubhamkumar108/protein"   # adjust if path differs

# 2a. Protein index map
with open(f"{BASE}/protein_index_map.json") as f:
    protein_index_map = json.load(f)
idx_to_pid    = {v: k for k, v in protein_index_map.items()}
N_PROTEINS    = len(protein_index_map)
valid_indices = set(range(N_PROTEINS))
print(f"Proteins in index : {N_PROTEINS}")

# 2b. Protein graphs
with open(f"{BASE}/proteinGraphsIndexed.pkl", "rb") as f:
    protein_graphs = pickle.load(f)
print(f"Protein graphs    : {len(protein_graphs)}")

# 2c. PPI edges
ppi_df = pd.read_csv(f"{BASE}/positiveEdges_indexed.csv")
src, dst = ppi_df["Node1"].tolist(), ppi_df["Node2"].tolist()
ppi_edge_index = torch.tensor([src+dst, dst+src], dtype=torch.long).to(device)
print(f"PPI edges (bi)    : {ppi_edge_index.shape[1]}")

# 2d. Positive complexes
with open(f"{BASE}/indexed_complexes.json") as f:
    pos_complexes_raw = json.load(f)
pos_complexes = [
    cx for cx in pos_complexes_raw
    if len(cx) >= 2 and all(i in valid_indices for i in cx)
]
print(f"Positive complexes: {len(pos_complexes)}")

# 2e. Negative complexes (from file, then top up with random)
with open(f"{BASE}/N_RANDOM_Comb.json") as f:
    neg_complexes_raw = json.load(f)

neg_complexes = []
for cx in neg_complexes_raw:
    mapped, skip = [], False
    for m in cx:
        key = str(m).strip()
        if key in protein_index_map:
            mapped.append(protein_index_map[key])
        else:
            skip = True; break
    if not skip and len(mapped) >= 2:
        neg_complexes.append(mapped)
print(f"Neg from file     : {len(neg_complexes)}")

target   = len(pos_complexes)
pos_sets = set(frozenset(cx) for cx in pos_complexes)
all_idx  = list(valid_indices)
rng      = random.Random(SEED)
while len(neg_complexes) < target:
    size  = rng.choice([2, 3, 4])
    combo = rng.sample(all_idx, size)
    if frozenset(combo) not in pos_sets:
        neg_complexes.append(combo)
if len(neg_complexes) > target:
    neg_complexes = random.sample(neg_complexes, target)
print(f"Final neg complexes: {len(neg_complexes)}")


In [ ]:
# ============================================================
# CELL 3 — Train / Val / Test split
# ============================================================
def split_list(lst, label, train_r=0.70, val_r=0.15, seed=42):
    data = [(cx, label) for cx in lst]
    random.Random(seed).shuffle(data)
    n = len(data); t1 = int(n*train_r); t2 = int(n*(train_r+val_r))
    return data[:t1], data[t1:t2], data[t2:]

pos_tr, pos_va, pos_te = split_list(pos_complexes, 1)
neg_tr, neg_va, neg_te = split_list(neg_complexes, 0)

train_data = pos_tr + neg_tr; random.shuffle(train_data)
val_data   = pos_va + neg_va; random.shuffle(val_data)
test_data  = pos_te + neg_te

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

# pos_weight for the loss (data is ~balanced, so this is ~1.0 — kept for safety)
n_pos = sum(lbl for _, lbl in train_data)
n_neg = len(train_data) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float, device=device)
print(f"pos_weight: {pos_weight.item():.3f}")


In [ ]:
# ============================================================
# CELL 4 — Models  (simplified, line-by-line)
# ============================================================

class GAT1(nn.Module):
    """Structure encoder: one protein graph -> [1, hidden_dim]."""

    def __init__(self, input_dim=24, hidden_dim=HIDDEN_DIM, heads=HEADS):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv3 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.pool1 = SAGPooling(hidden_dim)
        self.pool2 = SAGPooling(hidden_dim)
        self.pool3 = SAGPooling(hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)

    def forward(self, data):
        x  = data.x.float()
        ei = data.edge_index
        # Every protein graph always has edge_attr (residue distance < 5 angstrom),
        # so the "is not None" check was dead code. We just use it directly.
        ea = data.edge_attr.float()
        b  = data.batch

        x = self.fc1(x)

        # ---- Block 1 (no residual) ----
        x = self.conv1(x, ei, edge_attr=ea)
        x = self.bn1(x)
        x = F.relu(x)
        x, ei, ea, b, _, _ = self.pool1(x, ei, edge_attr=ea, batch=b)

        # ---- Block 2 (no residual) ----
        x = self.conv2(x, ei, edge_attr=ea)
        x = self.bn2(x)
        x = F.relu(x)
        x, ei, ea, b, _, _ = self.pool2(x, ei, edge_attr=ea, batch=b)

        # ---- Block 3 (no residual) ----
        x = self.conv3(x, ei, edge_attr=ea)
        x = self.bn3(x)
        x = F.relu(x)
        x, ei, ea, b, _, _ = self.pool3(x, ei, edge_attr=ea, batch=b)

        return global_mean_pool(x, b)   # [B, hidden_dim]


class GCN_Refiner(nn.Module):
    """2-layer GCN that refines protein embeddings with PPI context. 64 -> 64 throughout."""

    def __init__(self, in_dim=HIDDEN_DIM, out_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.gcn1 = GCNConv(out_dim, out_dim)
        self.gcn2 = GCNConv(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)

    def forward(self, X, ppi_edge_index):
        X = self.proj(X)

        X = self.gcn1(X, ppi_edge_index)
        X = F.relu(X)
        X = self.bn1(X)

        X = self.gcn2(X, ppi_edge_index)
        X = F.relu(X)
        X = self.bn2(X)
        return X                          # [N, EMBED_DIM]


class ComplexPredictor(nn.Module):
    """Mean-pool member embeddings -> MLP -> binary logit."""

    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64), nn.ReLU(),
            nn.Linear( 64,   1)
        )

    def forward(self, H, complex_indices_batch):
        logits = []
        for members in complex_indices_batch:
            idx    = torch.tensor(members, dtype=torch.long, device=H.device)
            pooled = H[idx].mean(0)
            logits.append(self.mlp(pooled))
        return torch.cat(logits, dim=0)


In [ ]:
# ============================================================
# CELL 5 — Instantiate + ONE joint optimizer
# ============================================================
gat_enc   = GAT1().to(device)
gcn_ref   = GCN_Refiner().to(device)
predictor = ComplexPredictor().to(device)

# Single optimizer over ALL three models -> all of them learn together.
optimizer = torch.optim.Adam(
    list(gat_enc.parameters()) +
    list(gcn_ref.parameters()) +
    list(predictor.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

n_params = (sum(p.numel() for p in gat_enc.parameters()) +
            sum(p.numel() for p in gcn_ref.parameters()) +
            sum(p.numel() for p in predictor.parameters()))
print(f"Total parameters: {n_params:,}")


In [ ]:
# ============================================================
# CELL 6 — Helpers: encode all proteins, and evaluate
# ============================================================

def encode_all_proteins():
    """GAT over every protein, in chunks, WITH gradients -> [N, HIDDEN_DIM]."""
    embs = []
    for i in range(0, len(protein_graphs), GAT_BATCH):
        chunk = protein_graphs[i:i + GAT_BATCH]
        batch = Batch.from_data_list(chunk).to(device)
        embs.append(gat_enc(batch))          # keeps grad graph
    return torch.cat(embs, dim=0)

@torch.no_grad()
def compute_H_eval():
    """Same as above but no grad -> protein embeddings refined by GCN, for eval."""
    gat_enc.eval(); gcn_ref.eval()
    embs = []
    for i in range(0, len(protein_graphs), GAT_BATCH):
        chunk = protein_graphs[i:i + GAT_BATCH]
        batch = Batch.from_data_list(chunk).to(device)
        embs.append(gat_enc(batch))
    x = torch.cat(embs, dim=0)
    return gcn_ref(x, ppi_edge_index)

@torch.no_grad()
def evaluate(data_list, H):
    gcn_ref.eval(); predictor.eval()
    members = [cx  for cx, _   in data_list]
    Y       = np.array([lbl for _, lbl in data_list], dtype=float)
    L       = predictor(H, members).cpu().numpy()
    P       = 1.0 / (1.0 + np.exp(-L))
    pred    = (P >= 0.5).astype(int)
    return {
        "loss"     : float(F.binary_cross_entropy_with_logits(
                         torch.tensor(L), torch.tensor(Y)).item()),
        "auc"      : roc_auc_score(Y, P),
        "ap"       : average_precision_score(Y, P),
        "acc"      : accuracy_score(Y, pred),
        "f1"       : f1_score(Y, pred, zero_division=0),
        "precision": precision_score(Y, pred, zero_division=0),
        "recall"   : recall_score(Y, pred, zero_division=0),
    }


In [ ]:
# ============================================================
# CELL 7 — Joint training loop (all 3 models, 200 epochs, save best)
# ============================================================
CKPT     = "/kaggle/working/best_complex_model.pt"
best_auc = 0.0
history  = []

# All training complexes + labels (full-batch, one backward per epoch)
train_members = [cx  for cx, _   in train_data]
train_labels  = torch.tensor([lbl for _, lbl in train_data],
                             dtype=torch.float, device=device)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    gat_enc.train(); gcn_ref.train(); predictor.train()
    optimizer.zero_grad()

    # forward:  GAT -> GCN -> predictor   (gradients flow through ALL three)
    x_struct = encode_all_proteins()                 # [N, 64]  with grad
    H        = gcn_ref(x_struct, ppi_edge_index)     # [N, 64]
    logits   = predictor(H, train_members)           # [n_train]

    loss = F.binary_cross_entropy_with_logits(logits, train_labels, pos_weight=pos_weight)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(gat_enc.parameters()) +
        list(gcn_ref.parameters()) +
        list(predictor.parameters()), 1.0)
    optimizer.step()
    scheduler.step()

    # ---- validation ----
    H_val = compute_H_eval()
    val_m = evaluate(val_data, H_val)
    history.append({"epoch": epoch, "train_loss": loss.item(),
                    **{f"val_{k}": v for k, v in val_m.items()}})

    if epoch % 20 == 0 or epoch == 1:
        print(f"Ep {epoch:3d} | loss={loss.item():.4f} | "
              f"auc={val_m['auc']:.4f} | f1={val_m['f1']:.4f} | "
              f"acc={val_m['acc']:.4f} | {time.time()-t0:.1f}s")

    # ---- save best (all three models) ----
    if val_m["auc"] > best_auc:
        best_auc = val_m["auc"]
        torch.save({"gat_enc"  : gat_enc.state_dict(),
                    "gcn_ref"  : gcn_ref.state_dict(),
                    "predictor": predictor.state_dict(),
                    "epoch"    : epoch,
                    "val_auc"  : best_auc}, CKPT)

    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nBest val AUC: {best_auc:.4f}  ->  {CKPT}")


In [ ]:
# ============================================================
# CELL 8 — Test evaluation (loads best checkpoint)
# ============================================================
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
gat_enc.load_state_dict(ckpt["gat_enc"])
gcn_ref.load_state_dict(ckpt["gcn_ref"])
predictor.load_state_dict(ckpt["predictor"])
print(f"Loaded best  epoch={ckpt['epoch']}  val_auc={ckpt['val_auc']:.4f}")

H_test = compute_H_eval()
test_m = evaluate(test_data, H_test)
print("\n===== TEST RESULTS =====")
for k, v in test_m.items():
    print(f"  {k:12s}: {v:.4f}")


In [ ]:
# ============================================================
# CELL 9 — Inference helper
# ============================================================
def predict_complex(member_indices, threshold=0.5):
    gat_enc.eval(); gcn_ref.eval(); predictor.eval()
    with torch.no_grad():
        H     = compute_H_eval()
        logit = predictor(H, [member_indices])
        prob  = torch.sigmoid(logit).item()
    label = "COMPLEX" if prob >= threshold else "NOT COMPLEX"
    names = [idx_to_pid.get(i, str(i)) for i in member_indices]
    print(f"{names}  ->  prob={prob:.4f}  ->  {label}")
    return prob

# Example:
# predict_complex([7442, 2193, 2580])
# predict_complex([0, 100, 200])


In [ ]:
# ============================================================
# CELL 10 — Plot training curves
# ============================================================
import matplotlib.pyplot as plt
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df.epoch, df.train_loss, label="train")
axes[0].plot(df.epoch, df.val_loss,   label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(df.epoch, df.val_auc, label="AUC")
axes[1].plot(df.epoch, df.val_f1,  label="F1")
axes[1].set_title("Val Metrics"); axes[1].legend()
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()
